# Project: read handwritten digits
**Machine Learning · Week 5 (Neural networks with PyTorch) · Day 5**

Lesson page: https://sara-aminpour.com/tutorials/ml/w5d5.html

**Project:** a neural network that reads handwritten digits. The data: 1,797 tiny 8 x 8 images
from scikit-learn, written by 43 people. Everything from this week comes together: a network,
backpropagation, mini-batches, Adam and early stopping.

In [ ]:
import copy
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

torch.set_num_threads(1)
digits = load_digits()
print("images:", digits.images.shape, " as feature vectors:", digits.data.shape)
print("pixel values from", digits.data.min(), "to", digits.data.max())

fig, axes = plt.subplots(2, 8, figsize=(8, 2.4), layout="constrained")
for ax, image, label in zip(axes.ravel(), digits.images, digits.target):
    ax.imshow(image, cmap="Purples")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Split three ways
Training points to learn from, validation points for early stopping, and test points we open once.

In [ ]:
X = digits.data / 16.0                     # pixel values from 0 to 1
y = digits.target
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_rest, y_rest, test_size=0.25, random_state=0, stratify=y_rest)
print("train", len(X_train), "| validation", len(X_val), "| test", len(X_test))

T = lambda a: torch.tensor(a, dtype=torch.float32)
L = lambda a: torch.tensor(a, dtype=torch.long)   # class labels are whole numbers
Xtr, Xva, Xte = T(X_train), T(X_val), T(X_test)
ytr, yva, yte = L(y_train), L(y_val), L(y_test)

## Train with every habit from this week
64 inputs (one per pixel), 64 hidden ReLU neurons, 10 outputs (one score per digit).
`CrossEntropyLoss` applies the softmax for us (week 3).

In [ ]:
def make_model(hidden=64, seed=0):
    torch.manual_seed(seed)
    return torch.nn.Sequential(torch.nn.Linear(64, hidden), torch.nn.ReLU(), torch.nn.Linear(hidden, 10))

loss_fn = torch.nn.CrossEntropyLoss()

def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()

def fit(model, patience=10, max_epochs=300, verbose=False):
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    loader = DataLoader(TensorDataset(Xtr, ytr), batch_size=32, shuffle=True, generator=torch.Generator().manual_seed(0))
    best_loss, best_state, best_epoch, waited = float("inf"), None, 0, 0
    for epoch in range(max_epochs):
        for xb, yb in loader:                      # mini-batches
            opt.zero_grad()
            loss_fn(model(xb), yb).backward()      # backpropagation
            opt.step()                             # Adam
        with torch.no_grad():
            val_loss = loss_fn(model(Xva), yva).item()
        if val_loss < best_loss:                   # early stopping
            best_loss, best_state, best_epoch, waited = val_loss, copy.deepcopy(model.state_dict()), epoch, 0
        else:
            waited += 1
            if waited >= patience:
                break
        if verbose and epoch % 20 == 0:
            print(f"epoch {epoch:3d}: validation loss {val_loss:.3f}, validation accuracy {accuracy(model, Xva, yva):.3f}")
    model.load_state_dict(best_state)
    if verbose:
        print(f"stopped after epoch {epoch}; kept the weights from epoch {best_epoch}")
    return model

model = fit(make_model(), verbose=True)

## The final exam
Accuracy on the 360 test digits, and the **confusion matrix**: row = the true digit, column = the guess.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

print(f"test accuracy: {accuracy(model, Xte, yte):.3f}")
with torch.no_grad():
    guess = model(Xte).argmax(dim=1).numpy()

fig, ax = plt.subplots(figsize=(4.6, 4.2), layout="constrained")
ConfusionMatrixDisplay.from_predictions(y_test, guess, cmap="Purples", colorbar=False, ax=ax)
ax.set_xlabel("the network's guess")
ax.set_ylabel("true digit")
ax.grid(False)
plt.show()

wrong = np.where(guess != y_test)[0]
rows = (len(wrong) + 5) // 6
fig, axes = plt.subplots(rows, 6, figsize=(6.6, 1.5 * rows), layout="constrained", squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for ax, i in zip(axes.ravel(), wrong):
    ax.imshow(X_test[i].reshape(8, 8), cmap="Purples")
    ax.set_title(f"{y_test[i]} → {guess[i]}", fontsize=11)
plt.show()
print(f"{len(wrong)} mistakes, written as true → guess. Would you have read them correctly?")

## Your turn: how small can the network be?
Try hidden layers of 2, 4, 8, 16 and 32 neurons. Choose with the **validation** accuracy, then open
the test set once, for your final choice only.

In [ ]:
results = {}
for hidden in [2, 4, 8, 16, 32]:
    pass   # TODO: m = fit(make_model(hidden)); results[hidden] = accuracy on the validation set
print(results)
# TODO: pick the smallest size within 1% of the best validation accuracy, and report its test accuracy

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
results = {}
for hidden in [2, 4, 8, 16, 32, 64]:
    m = fit(make_model(hidden))
    results[hidden] = accuracy(m, Xva, yva)
    n_weights = sum(p.numel() for p in m.parameters())
    print(f"{hidden:2d} hidden neurons ({n_weights:5,d} weights): validation accuracy {results[hidden]:.3f}")

best = max(results.values())
choice = min(h for h, acc in results.items() if acc >= best - 0.01)
final = fit(make_model(choice))
print(f"smallest size within 1% of the best: {choice} neurons; its test accuracy (opened once): {accuracy(final, Xte, yte):.3f}")